### Tokenizing text

In [ ]:
# Download The Verdict from Raschka's GitHub repo
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-virdict.txt', <http.client.HTTPMessage at 0x78297eea0980>)

In [ ]:
# Open the file, count the number of characters and look at the first
# 100 characters
with open("the-verdict.txt", 'r', encoding='utf-8') as f:
  raw_text = f.read()
print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [ ]:
# Let's start by only splitting the text on whitespaces
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [ ]:
# Then we include simple punctuations such as commas and periods
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [ ]:
# Remove whitespaces from token list.
# NOTE: tokenizers for LLMs keep whitespaces as they may be important
# (e.g. Python uses indentations to delimit code blocks)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [ ]:
# Let's add more punctuation to the regex rule for splitting
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [ ]:
# Tokenize the full text of The Verdict
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### Converting tokens into token IDs

In [ ]:
# We need to build a vocabulary that maps each token to a token ID
# First we get the unique tokens in the text to build the vocabulary
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [ ]:
# Now we actualy create the vocabulary that maps each token to a
# unique ID.
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [ ]:
# Let's now build a more structured tokenizer class in Python
# The class has two methods: encode(self, text) that encodes text into
# token IDs and decode(self, ids) which turns token IDs back to text

class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i: s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = ' '.join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # remove spaces before specified punctuation
    return text

In [ ]:
# Now we can instantiate new tokenizer objects using a vocabulary
# and use the object's method to encode and decode text
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
          Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [ ]:
# This will raise an error since the word 'Hello' was not used in
# The Verdict. This highlights the need to consider large and diverse
# training sets to extend the vocabulary when working on LLMs.
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

### Adding special context tokens

In [ ]:
# Let's further refine our simple tokenizer to include two more special
# contecxt tokens: '<|endoftext|>' and '<|unk|>'. While '<|unk|>' will
# allow the tokenizer to deal with tokens that were not part of the
# training set (and thus are not part of our vocabolary),
# '<|endoftext|>' is a special token used to delimit different and 
# text unrelated text sources such as documents, books etc. This is
# a common thing to do when training GPT-like LLMs on multiple independent
# documents or books because it helps the LLM understand that although
# these text sources are concatenated for training, they are, in fact,
# unrelated
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [ ]:
# Now we can update our SimpleTokenizer class to reflect the changes
# in the vocabulary
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i: s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [item if item in self.str_to_int else "<|unk|>"
                    for item in preprocessed]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = ' '.join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
# Test the new SimpleTokenizer class with two sample unrelated text
# sources
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [ ]:
# The tokenizer correctly handles both the <|endoftext|> delimitation
# mark as well as unknown tokens such as 'Hello'
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))


[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Byte Pair Encoding

In [ ]:
# Now we will move to a more sophisticated and efficient tokenizer
# called *byte pair encoding tokenizer*, which breaks down into
# subwords units. This was used by both  GPT-2 and GPT-3 and it does
# not have an '<|unk|>' token for out-of-vocabulary tokens.
# To use the *byte pair encoding tokenizer* we need to install the
# tiktoken package

# If using pip
!pip install tiktoken==0.7.0

# If using uv
!uv add tiktoken==0.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.1 MB/s eta 0:00:00
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.13.0
    Uninstalling tiktoken-0.13.0:
      Successfully uninstalled tiktoken-0.13.0


In [ ]:
# Ensure the version of the package is 0.7.0 to ensure
# reproducibility
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.7.0


In [ ]:
# Creating a new BPE tokenizer object is as simple as this
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# The API of this tokenizer object is similar to what we have
# implemented in the SimpleTokenizer classes, with the encode() and
# decode() methods
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
        "of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


### Data sampling with sliding window

In [ ]:
# The next step in creating the embeddings for the LLM is to generate
# the input–target pairs required for training an LLM. We begin by
# tokenizing The Verdict using the BPE tokenizer
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))
# Remove he first 50 tokens from the dataset for demonstration 
# purposes, as it results in a slightly more interesting text
#  passage in the next steps
enc_sample = enc_text[50:]

5145


In [ ]:
# One of the easiest and most intuitive ways to create the 
# input–target pairs for the next-word prediction task is to create
# two variables, x and y, where x contains the input tokens and y
# contains the targets, which are the inputs shifted by 1:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:       {y}")

x: [290, 4920, 2241, 287]
y:        [4920, 2241, 287, 257]


In [ ]:
# Everything left of the arrow (---->) refers to the input an LLM 
# would receive, and the token ID on the right side of the arrow 
# represents the target token ID that the LLM is supposed to predict. 
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "--->", desired)


for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257
 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


In [ ]:
# To efficiently load all the input and target vectors we need to
# construct a PyTorch Dataset 
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)
    for i in range(0, len(token_ids) - max_length, stride): # Uses a sliding window to chunk the book into overlapping sequences of max_length
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self): # returns the total number of rows in the dataset
    return len(self.input_ids)

  def __getitem__(self, index): # returns a single row from the dataset
    return self.input_ids[index], self.target_ids[index]

In [ ]:
# Now that we have defined a dataset, we can define the dataloader
# that returns the pairs of inputs and targets in batches
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
    ):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers,
  )
  return dataloader



In [ ]:
# Let's test a dataloader with a batch_size of 1 and a context size
# of 4 to develop an intuition of how the Dataset and DataLoader
# classes work together
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [39]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
# Let’s look briefly at how we can use the data loader to sample with
# a batch size greater than 1
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs\n", inputs)
print("\nTargets\n", targets)

Inputs
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating Token Embeddings

In [ ]:
# Let's see how the token ID to embedding vector conversion works
# with a hands on example. For the sake of simplicity, suppose we
# have a small vocabulary of only 6 words, and we want to create
# embeddings of size 3
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# We can see that the matrix has siz rows and three columns, i.e. 
# one row for each possible token in the vocabulary and one column
# for each of the embedding's dimensions
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
# Let's now apply it to a token ID to obtain the embedding vector.
# We can see that the returned vector is the 4th (0-index) row in 
# the embedding layer weight matrix. The embedding layer is in fact,
# a simple indexing operation!
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


**Note**

For those who are familiar with one-hot encoding, the embedding layer approach described here is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully connected layer.

In [ ]:
# Let's now apply the embedding layer to all the input IDs
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Encoding word position

In [ ]:
# Having now created embedding vectors from token IDs, next we’ll add
# a small modification to these embedding vectors to encode positional
# information about a token within a text. This is necessary as the
# attention mechanism of LLMs does not have a notion of position for
# tokens in a sequence. By encoding positional information, the same
# token which appears in different positions inside the sequence will
# result in a slightly different embedding representation
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text,
                                  batch_size=8,
                                  max_length=max_length,
                                  stride=max_length,
                                  shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs: \n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
# 8 batches each consisting of 4 tokens each represented by a
# 256-dimensional embedding vector
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
# For a GPT model’s absolute embedding approach, we just need to create
# another embedding layer that has the same embedding dimension as
# the token_embedding_ layer
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length)) # embedding each possible position index
print(pos_embeddings.shape)

torch.Size([4, 256])


In [ ]:
# We the add the positional embeddings to the token embeddings to
# obtain the final input embeddings ready to be fed to our LLM !
input_embeddings = token_embeddings + pos_embeddings # this works well thanks to broadcasting rules
print(input_embeddings.shape)

torch.Size([8, 4, 256])
